# Heart Disease Prediction using Logistic Regression

This project predicts the presence of heart disease from clinical measurements. The notebook follows a complete machine-learning workflow: understanding the data, cleaning it, testing feature associations, building a pipeline, tuning Logistic Regression, selecting a threshold, and evaluating the final model.

## 1. Imports and setup

The following libraries are used for data handling, visualisation, statistical testing, model training, and evaluation.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import chi2_contingency, pearsonr
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_validate, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='deep')
RANDOM_STATE = 42

## 2. Load and inspect the dataset

HeartDisease is the target column. A value of 0 represents no heart disease and 1 represents heart disease.

In [ ]:
df = pd.read_csv('heart.csv')
target = 'HeartDisease'
 

print(f'Dataset shape: {df.shape}')
display(df.head())
display(df.dtypes.to_frame('Data type'))
display(df.isna().sum().to_frame('Missing values'))

## 3. Exploratory data analysis

The plots below help us understand the distribution of key numerical health measurements before preprocessing.

In [ ]:
numerical_columns = ['Age', 'RestingBP', 'Cholesterol', 'MaxHR']
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for ax, column in zip(axes.ravel(), numerical_columns):
    sns.histplot(data=df, x=column, kde=True, ax=ax)
    ax.set_title(f'Distribution of {column}')

plt.tight_layout()
plt.show()

plt.figure(figsize=(5, 4))
sns.countplot(data=df, x=target)
plt.title('Distribution of Heart Disease')
plt.xlabel('Heart Disease (0 = No, 1 = Yes)')
plt.ylabel('Number of patients')
plt.show()

## 4. Data cleaning

Duplicate rows are removed. Zero is not a realistic value for cholesterol or resting blood pressure in this dataset, so those values are replaced with the mean of the non-zero values.

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)

df.loc[df['Cholesterol'] == 0, 'Cholesterol'] = df.loc[df['Cholesterol'] != 0, 'Cholesterol'].mean().astype(int)
df.loc[df['RestingBP'] == 0, 'RestingBP'] =  df.loc[df['RestingBP'] != 0, 'RestingBP'].mean().astype(int)

print(f'Cleaned dataset shape: {df.shape}')

In [ ]:
df.head()

## 5. Encode categorical features

Logistic Regression needs numeric input. One-hot encoding converts the categorical clinical features into binary columns while avoiding redundant columns.

In [ ]:
categorical_columns = ['Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope']
model_data = pd.get_dummies(df, columns=categorical_columns, drop_first=True, dtype=int)

print(f'Encoded dataset shape: {model_data.shape}')
display(model_data.head())

## 6. Chi-square test for categorical features

This test measures the association between each encoded categorical feature and heart disease. A larger chi-square statistic and a smaller p-value indicate a stronger association.

In [ ]:
features = [
    'FastingBS',
    'Sex_M',
    'ChestPainType_ATA',
    'ChestPainType_NAP',
    'ChestPainType_TA',
    'RestingECG_Normal',
    'RestingECG_ST',
    'ExerciseAngina_Y',
    'ST_Slope_Flat',
    'ST_Slope_Up'
]
chi2_rows = []

for feature in  features:
    table = pd.crosstab(model_data[feature], model_data[target])
    chi2_stat, p_value, degrees_of_freedom, _ = chi2_contingency(table)
    chi2_rows.append({
        'Feature': feature,
        'Chi-Square Score': chi2_stat,
        'P-Value': p_value,
        'Decision (alpha = 0.05)': 'Keep' if p_value < 0.05 else 'Review'
    })

chi2_results = pd.DataFrame(chi2_rows).sort_values('Chi-Square Score', ascending=False).reset_index(drop=True)
display(chi2_results.style.format({'Chi-Square Score': '{:.3f}', 'P-Value': '{:.4g}'}))

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(data=chi2_results, x='Chi-Square Score', y='Feature', hue='Feature', legend=False)
plt.title('Chi-Square Association with Heart Disease')
plt.xlabel('Chi-Square Score')
plt.ylabel('Encoded categorical feature')
plt.tight_layout()
plt.show()

## 7. Correlation of numerical features

Pearson correlation is used to examine the linear relationship between numerical measurements and the target.

In [ ]:
numeric_model_features = ['Age', 'RestingBP', 'Cholesterol', 'FastingBS', 'MaxHR', 'Oldpeak']
correlation_results = pd.DataFrame({
    'Feature': numeric_model_features,
    'Correlation': [pearsonr(model_data[feature], model_data[target])[0] for feature in numeric_model_features]
}).sort_values('Correlation', key=lambda values: values.abs(), ascending=False)

display(correlation_results.style.format({'Correlation': '{:.3f}'}))

plt.figure(figsize=(8, 4))
sns.barplot(data=correlation_results, x='Feature', y='Correlation', hue='Feature', legend=False)
plt.title('Correlation of Numerical Features with Heart Disease')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()

## 8. Define features and create train-test sets

All engineered predictor columns are used. The test set is separated before model selection and remains untouched until the final evaluation.

In [ ]:
X = model_data.drop(columns=target)
y = model_data[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print(f'Training set: {X_train.shape}')
print(f'Test set: {X_test.shape}')

## 9. Build the Logistic Regression pipeline

Feature scaling is placed inside the pipeline so it is learned only from the appropriate training folds. This prevents data leakage during cross-validation.

In [ ]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(max_iter=5000, random_state=RANDOM_STATE))
])

pipeline

## 10. Tune hyperparameters with GridSearchCV

GridSearchCV tests compatible solver, penalty, and regularisation combinations. The model is selected using F1 score, which balances precision and recall.

In [ ]:
param_grid = [
    {'logreg__solver': ['lbfgs'], 'logreg__penalty': ['l2'], 'logreg__C': [0.01, 0.1, 1, 10, 100]},
    {'logreg__solver': ['liblinear'], 'logreg__penalty': ['l1', 'l2'], 'logreg__C': [0.01, 0.1, 1, 10, 100]},
    {'logreg__solver': ['saga'], 'logreg__penalty': ['l1', 'l2'], 'logreg__C': [0.01, 0.1, 1, 10, 100]},
    {'logreg__solver': ['saga'], 'logreg__penalty': ['elasticnet'], 'logreg__C': [0.01, 0.1, 1, 10, 100], 'logreg__l1_ratio': [0.2, 0.5, 0.8]}
]
scoring = {'accuracy': 'accuracy', 'precision': 'precision', 'recall': 'recall', 'f1': 'f1', 'roc_auc': 'roc_auc'}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

grid = GridSearchCV(
    estimator=pipeline, param_grid=param_grid, scoring=scoring, refit='f1',
    cv=cv, n_jobs=-1, return_train_score=False
)
grid.fit(X_train, y_train)

print('Best parameters:', grid.best_params_)
print(f'Best cross-validated F1 score: {grid.best_score_:.4f}')

In [ ]:
cv_results = pd.DataFrame(grid.cv_results_)
result_columns = ['rank_test_f1', 'mean_test_accuracy', 'mean_test_precision', 'mean_test_recall', 'mean_test_f1', 'mean_test_roc_auc', 'params']
display(cv_results[result_columns].sort_values('rank_test_f1').head(10))

## 11. Cross-validation check

A multi-metric cross-validation check shows how consistently the selected pipeline performs on the training data.

In [ ]:
best_model = grid.best_estimator_
validation_scores = cross_validate(best_model, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)

cv_summary = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC'],
    'Mean': [validation_scores['test_accuracy'].mean(), validation_scores['test_precision'].mean(), validation_scores['test_recall'].mean(), validation_scores['test_f1'].mean(), validation_scores['test_roc_auc'].mean()],
    'Standard Deviation': [validation_scores['test_accuracy'].std(), validation_scores['test_precision'].std(), validation_scores['test_recall'].std(), validation_scores['test_f1'].std(), validation_scores['test_roc_auc'].std()]
})
display(cv_summary.style.format({'Mean': '{:.4f}', 'Standard Deviation': '{:.4f}'}))

## 12. Threshold optimisation

The default threshold is 0.50, but it may not be the best balance of precision and recall. The threshold is selected using out-of-fold predictions from the training data, not the test data.

In [ ]:
train_probabilities = cross_val_predict(best_model, X_train, y_train, cv=cv, method='predict_proba', n_jobs=-1)[:, 1]
threshold_rows = []

for threshold in np.arange(0.05, 0.96, 0.01):
    threshold_predictions = (train_probabilities >= threshold).astype(int)
    threshold_rows.append({
        'Threshold': threshold,
        'Precision': precision_score(y_train, threshold_predictions, zero_division=0),
        'Recall': recall_score(y_train, threshold_predictions, zero_division=0),
        'F1 Score': f1_score(y_train, threshold_predictions, zero_division=0)
    })

threshold_results = pd.DataFrame(threshold_rows)
best_threshold = threshold_results.loc[threshold_results['F1 Score'].idxmax(), 'Threshold']
print(f'Chosen F1-optimal threshold: {best_threshold:.2f}')
display(threshold_results.sort_values('F1 Score', ascending=False).head())

In [ ]:
plt.figure(figsize=(9, 4))
for metric in ['Precision', 'Recall', 'F1 Score']:
    plt.plot(threshold_results['Threshold'], threshold_results[metric], label=metric)
plt.axvline(best_threshold, color='black', linestyle='--', label=f'Chosen threshold = {best_threshold:.2f}')
plt.title('Threshold Optimisation on Training Data')
plt.xlabel('Classification threshold')
plt.ylabel('Score')
plt.legend()
plt.tight_layout()
plt.show()

## 13. Final evaluation on the test set

The selected model is fitted on the full training set and evaluated once on the unseen test set. ROC-AUC uses predicted probabilities; the remaining metrics use the selected threshold.

In [ ]:
best_model.fit(X_train, y_train)
test_probabilities = best_model.predict_proba(X_test)[:, 1]
test_predictions = (test_probabilities >= best_threshold).astype(int)

final_metrics = pd.Series({
    'Accuracy': accuracy_score(y_test, test_predictions),
    'Precision': precision_score(y_test, test_predictions, zero_division=0),
    'Recall': recall_score(y_test, test_predictions, zero_division=0),
    'F1 Score': f1_score(y_test, test_predictions, zero_division=0),
    'ROC-AUC': roc_auc_score(y_test, test_probabilities)
})

display(final_metrics.to_frame('Test Score').style.format({'Test Score': '{:.4f}'}))
print(classification_report(y_test, test_predictions, digits=4))

In [ ]:
cm = confusion_matrix(y_test, test_predictions)
plt.figure(figsize=(5, 4))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues', cbar=False,
    xticklabels=['No disease', 'Heart disease'],
    yticklabels=['No disease', 'Heart disease']
)
plt.title(f'Confusion Matrix (threshold = {best_threshold:.2f})')
plt.xlabel('Predicted label')
plt.ylabel('True label')
plt.tight_layout()
plt.show()

## Conclusion

This notebook demonstrates a structured Logistic Regression workflow for heart disease prediction. The next development step is to save the trained pipeline and connect it to a simple user interface.